<!-- CHECK_ALLOWLIST: descriptive-posture banner cell — spec §9 / §7-quoting block, exempt per plan 0.13 -->
# E10 GSPS v0.4 — 01 Simulator Calibration

DESCRIPTIVE POSTURE — E10 GSPS v0.4 is a descriptive / illustrative iteration. It makes NO inferential beta claim. The deliverable is the FX-variance-share sensitivity surface (a description), not a verdict point. The §7 sign gate is a necessary-not-sufficient descriptive gate, not an inferential test (spec v0.4 §7 field 5 / §9). <!-- CHECK_ALLOWLIST -->

<!-- CHECK_ALLOWLIST: pre-pin cell quotes spec v0.4 §7 verbatim — exempt per plan 0.13 -->
## Pre-pin record (LOCKED)

E10 v0.4 PRE-PIN LOCKED (spec v0.4 §7) — seven fields declared BEFORE any simulation run; no post-data tuning.

| # | Field | Value |
|---|---|---|
| 1 | Sign | beta > 0 on the vol-on-vol regression — necessary-not-sufficient descriptive gate (§4.1). |
| 2 | Primary object | The FX-variance-share sensitivity surface (§4.3) plus the ex-ante-pinned break-even threshold (§4.4). |
| 3 | Material threshold | The hedge's break-even share (§4.4), derived ex ante from option-premium economics. No post-hoc adjustment. |
| 4 | Lag | Contemporaneous, monthly. |
| 5 | Posture | Descriptive / illustrative; demonstration-grade-or-below; NOT confirmatory. <!-- CHECK_ALLOWLIST --> |
| 6 | Panel & FE | (currency, month); 5 currencies (COP, BRL, EUR, GBP, NGN) x ~30 months ~= 150 cells; NGN post-June-2023 float; two-way FE (currency + month) primary and currency-FE-only co-primary (spec v0.6 CORRECTIONS-E10-6 W-1). |
| 7 | HALT condition | HALT on simulator-anchor failure / Q-variance dominance / surface uncomputable / spec-vs-data contradiction. |

## Purpose

Plan task 2.4 — the E10.1 calibration notebook. It documents the **R6 NHPP query-workflow simulation engine** (plan task 2.1) and the **calibrated representative-analyst Q-process** (task 2.2) against the NON-FANTASY observed-trace anchor.

It exhibits, with a trio HALT-checkpoint at each step:

1. the calibrated Q-process vs the observed anchor (`scratch/2026-05-20-e10-revision/e10_q_anchor_*`);
2. the **Cox / doubly-stochastic** overdispersion mechanism — Gamma-mixed sprint-intensity primary, log-normal-mixing sensitivity arm (Model QA Strong-1);
3. the five λ(t) modulations (calibration note §4);
4. the `Q_low` / `Q_high` band and the swept 60-130/mo Q envelope;
5. the two OPEN calibration items resolved as PARTIAL (`dispositions/E10.1_proxy_research_PARTIAL.md`).

Calibration source of record: `notebooks/e10_gsps/diagnostics/E10.1_calibration_note.md`.

## Decision-citation block — calibration data sources

> **Reference:** E10.1 calibration note §1-§2; `scratch/2026-05-20-e10-revision/e10_q_anchor_extraction.md` + `e10_q_anchor_{daily,buckets}.{csv,json}`.
> **Why:** the Q-process centre is anchored to the user's OWN observed Claude-Code data-query trace (the §6.2 NON-FANTASY clause) — not a fitted prior, not a synthetic generator.
> **Relevance to E10 v0.4:** Q is the ONLY simulated quantity; its calibration anchor sets the §4.3 surface's Q-volume range.
> **Connection to the deliverable:** the calibrated Q-process feeds the per-(currency, month) cost-stream trajectories from which the FX-variance-share surface is computed.

In [ ]:
# Trio discipline: every (why-markdown, code-cell, interpretation-
# markdown) trio is a HALT-checkpoint for human review per
# feedback_notebook_trio_checkpoint.md.
from __future__ import annotations

import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # headless execution (plan task 0.9)
import matplotlib.pyplot as plt
import numpy as np

from simulations.e10_gsps.modules.nhpp_engine import (
    NHPPSimulationEngineModule,
    calibrated_intensity_parameters,
    diurnal_hourly_envelope,
    simulate_monthly_query_counts,
)

REPO = Path.cwd()
while not (REPO / 'simulations').is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
ANCHOR = REPO / 'scratch' / '2026-05-20-e10-revision'

buckets = json.loads((ANCHOR / 'e10_q_anchor_buckets.json').read_text())
daily_rows = [
    ln.split(',')
    for ln in (ANCHOR / 'e10_q_anchor_daily.csv').read_text().splitlines()[1:]
]
centre_daily = [int(c) for s, d, c in daily_rows if s == 'centre']
bracket_daily = [int(c) for s, d, c in daily_rows if s == 'bracket']
print('centre daily counts :', centre_daily)
print('bracket daily counts:', bracket_daily)

## Interpretation

The observed anchor loads cleanly: 12 centre daily counts (abrigo-analytics) and 28 bracket daily counts (other projects). The trace is genuine and non-trivial — the RC pre-Phase-2 review cleared the spec §7 field-7a anchor-adequacy gate. No HALT on anchor grounds. Descriptive posture only.

## Decision-citation block — overdispersion mechanism (Cox)

> **Reference:** calibration note §3 (LOCK: Cox / doubly-stochastic NHPP) + §0 addendum A1/A2; Model QA pre-Phase-2 review (RATIFY).
> **Why:** the trace's daily variance-to-mean ratio ≈ 20 — reached INDEPENDENTLY in centre and bracket — is ~20x the equidispersed-Poisson value of 1. A pure NHPP is equidispersed conditional on λ(t) and cannot reproduce this; an explicit overdispersion mechanism is required. The Cox route ties Q-variance to an OBSERVED structured-burst pattern, not a free dispersion knob.
> **Relevance to E10 v0.4:** the mechanism inflates Var(Δlog Q) — the §4.3 denominator term competing with Var(Δlog FX).
> **Connection to the deliverable:** the Cox sprint-intensity layer is the overdispersion slot the R6 engine is built against.

**Variance-convention note (RC O-2):** the daily VMR figures use *sample* variance (n-1): centre 20.6, bracket 19.8. The *population* variance (n) recompute gives 18.9 / 19.1 — the same conclusion. The estimator choice does not move the verdict.

**Gamma-vs-log-normal (Model QA Strong-1):** the mechanism CLASS (Cox / doubly-stochastic) is trace-anchored and RATIFIED. The *Gamma* mixing law within that class is a conjugacy default — NOT discriminable from log-normal at n≈12-28 daily counts — so a log-normal-mixing **sensitivity arm** is carried alongside the Gamma primary.

In [ ]:
# VMR — sample and population convention, centre and bracket.
def vmr(counts, ddof):
    arr = np.asarray(counts, dtype=float)
    return arr.var(ddof=ddof) / arr.mean()

for label, counts in [('centre', centre_daily),
                      ('bracket', bracket_daily)]:
    print(f'{label:8s} daily VMR  sample(n-1)='
          f'{vmr(counts, 1):5.1f}  population(n)={vmr(counts, 0):5.1f}')
print('reported buckets.json day VMR: '
      f"centre={buckets['centre']['vmr']['day']:.1f} "
      f"bracket={buckets['bracket']['vmr']['day']:.1f}")

# The Cox engine reproduces decisive overdispersion in the monthly
# aggregate under all three locked mechanisms.
for mech in ('cox', 'cox_lognormal', 'negative_binomial'):
    params = calibrated_intensity_parameters(mechanism=mech)
    monthly = np.asarray(
        simulate_monthly_query_counts(
            params=params, n_months=240, seed=20260520),
        dtype=float,
    )
    fano = monthly.var(ddof=1) / monthly.mean()
    print(f'{mech:18s} monthly-Q  mean={monthly.mean():7.1f}  '
          f'Fano={fano:6.2f}  (equidispersed null = 1.00)')

## Interpretation

The observed daily VMR is ≈ 20 under both variance conventions and in both centre and bracket — decisive overdispersion. The Cox engine reproduces a monthly-aggregate Fano factor well above the equidispersed-Poisson value of 1 under the Gamma primary, the log-normal sensitivity arm, AND the §3.4 negative-binomial fallback. A pure NHPP could not — this confirms the calibration note §3 lock. Descriptive posture only.

## Decision-citation block — the five λ(t) modulations

> **Reference:** calibration note §4 (modulation forms locked) + spec §6.3.
> **Why:** the R6 intensity λ(t) is pre-committed to carry five modulations; their FUNCTIONAL FORMS are locked from the trace (modulations 1, 2, 3, 5) and — for modulation 4's magnitude — deferred to proxy research.
> **Relevance to E10 v0.4:** the modulations shape the within- and between-month structure of Var(Δlog Q).
> **Connection to the deliverable:** each modulation is a wired slot in the engine the §4.3 surface consumes.

In [ ]:
# Modulation 1 — the locked TWO-peak diurnal envelope (calib §4.1).
env = np.asarray(diurnal_hourly_envelope())
centre_diurnal = buckets['centre']['diurnal_utc']

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].bar(range(24), env, color='steelblue')
ax[0].set_title('Modulation 1: locked two-peak diurnal envelope')
ax[0].set_xlabel('UTC hour'); ax[0].set_ylabel('arrival weight')
obs_hours = sorted(int(h) for h in centre_diurnal)
ax[1].bar(obs_hours, [centre_diurnal[str(h)] for h in obs_hours],
          color='indianred')
ax[1].set_title('Observed centre diurnal (UTC) — bimodal')
ax[1].set_xlabel('UTC hour'); ax[1].set_ylabel('queries')
fig.tight_layout(); plt.close(fig)

peaks = [h for h in range(1, 23)
         if env[h] > env[h-1] and env[h] > env[h+1]]
print('diurnal envelope local maxima at UTC hours:', peaks)
print('observed centre diurnal clusters at UTC 11-12h and 21-22h '
      '-> a two-peak form is the correct lock')

modulations = {
    '1 diurnal+weekday': 'weekday weight x two-peak diurnal — LOCKED',
    '2 burstiness/overdisp': 'Cox Gamma-mixed sprint-intensity — '
                             'LOCKED (= the overdispersion mechanism)',
    '3 event spikes': 'sprint-intensity upper-tail spike mixture — '
                      'LOCKED',
    '4 month-end seasonality': 'end-of-month bump SHAPE wired; '
                               'MAGNITUDE OPEN -> PARTIAL',
    '5 secular drift': 'log-linear baseline-lambda trend — LOCKED',
}
for k, v in modulations.items():
    print(f'  {k:26s} {v}')

## Interpretation

Four of the five λ(t) modulations are own-trace-anchored and LOCKED: the two-peak diurnal envelope matches the observed bimodal UTC clustering; burstiness IS the Cox mechanism; event spikes are its upper tail; secular drift is a log-linear trend. Modulation 4's *magnitude* is the proxy-bracket-pending OPEN item (next trio). Descriptive posture only.

## Decision-citation block — calibrated Q-process vs anchor

> **Reference:** calibration note §2 (Q-volume range ~60-130/mo) + §0 addendum A3 (sweep the full envelope).
> **Why:** the calibrated centre must reproduce the anchored Q-volume band; Model QA Strong requires sweeping the WHOLE 60-130/mo envelope, not a single point.
> **Relevance to E10 v0.4:** the swept band IS the §4.3 surface's domain.
> **Connection to the deliverable:** the surface is evaluated on a grid across this band.

In [ ]:
# The calibrated Q-process — monthly-aggregate distribution vs the
# anchored ~60-130 queries/month centre band.
params = calibrated_intensity_parameters(mechanism='cox')
monthly = np.asarray(
    simulate_monthly_query_counts(
        params=params, n_months=600, seed=424242),
    dtype=float,
)
q_lo_band, q_hi_band = 60.0, 130.0
in_band = np.mean((monthly >= q_lo_band) & (monthly <= q_hi_band))
print(f'calibrated monthly-Q: mean={monthly.mean():.1f}  '
      f'median={np.median(monthly):.1f}  '
      f'p10={np.percentile(monthly,10):.0f}  '
      f'p90={np.percentile(monthly,90):.0f}')
print(f'fraction of months inside the anchored 60-130/mo band: '
      f'{in_band:.2f}')

fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.hist(monthly, bins=40, color='steelblue', alpha=0.85)
ax.axvspan(q_lo_band, q_hi_band, color='orange', alpha=0.25,
           label='anchored 60-130/mo centre band')
ax.set_xlabel('simulated monthly Q (priceable queries)')
ax.set_ylabel('months'); ax.legend()
ax.set_title('Calibrated Cox Q-process vs the anchored Q-volume band')
fig.tight_layout(); plt.close(fig)

# Q_high — pinned off the $399 Dune-Plus price, NOT trace-derived.
print(f'Q_high (spec §2.3, $399/$0.01) = {params.q_high:,.0f}/mo')
print(f'Q_low  (spec §2.4) = {params.q_low}  '
      '<- NaN: UNPINNED, proxy-bracket-pending (see next trio)')

## Interpretation

The calibrated Cox Q-process is centred on the anchored ~60-130 priceable-queries/month band — the representative analyst sits firmly in the spec §2.2 middle band, far below the `Q_high` = 39,900/mo Dune-Plus crossover. The §4.3 surface sweeps the whole envelope (Model QA Strong A3). `Q_high` is pinned off the $399 price, not the trace. `Q_low` is UNPINNED — the next trio. Descriptive posture only.

## Decision-citation block — the two OPEN items (PARTIAL)

> **Reference:** calibration note §0 addendum A5, §4.4, §5; plan task 2.2 / 2.3b proxy-research gate; CORR-E10P-8 / RC WEAK-2; disposition `dispositions/E10.1_proxy_research_PARTIAL.md`.
> **Why:** spec §6.2 routes bracket magnitudes to proxy research and §2.4 defers `Q_low` to a proxy-corroborated rule. The non-bypassable proxy-research data-fetch gate was evaluated.
> **Relevance to E10 v0.4:** the two items would have widened the §4.3 surface from a centre to a band on two dimensions.
> **Connection to the deliverable:** unpinned, the surface degrades from a band to a centre-only characterization on those dimensions — routed to PARTIAL per spec §9, never flat-filled.

In [ ]:
# The proxy-research gate outcome — the two OPEN calibration items.
# Both proxies are partially reachable but pin NO numeric value.
open_items = {
    'Q_low (qualitative-limit threshold, spec §2.4)': (
        'UNPINNED — the free tier qualitative limit is a CAPABILITY '
        'threshold (trace/archive/large-eth_getLogs/cross-protocol '
        'need), not a query-VOLUME threshold; The Graph publishes no '
        'per-second rate limit; the metadata-only trace cannot '
        'observe capability need. Carried as NaN.'),
    'month-end seasonality magnitude (modulation 4, §6.3)': (
        'UNPINNED — no public Graph/RPC/Dune statistic discloses a '
        'month-end reporting-cadence magnitude; own-trace too short. '
        'Engine default month_end_bump_magnitude = 0.0 (no-op).'),
}
for item, status in open_items.items():
    print(f'OPEN: {item}')
    print(f'      {status}\n')

# Verify the engine carries both honestly — no fabricated default.
p = calibrated_intensity_parameters()
assert np.isnan(p.q_low), 'Q_low must stay an unpinned NaN marker'
default_engine = NHPPSimulationEngineModule(params=p)
assert default_engine.month_end_bump_magnitude == 0.0
print('VERIFIED: Q_low=NaN (unpinned marker); month-end bump=0.0 '
      '(documented no-op). No flat-fill, no fabricated bracket.')
print('Disposition: dispositions/E10.1_proxy_research_PARTIAL.md '
      '-> PARTIAL on these two items (non-HALT, spec-anticipated).')

## Interpretation

The non-bypassable proxy-research gate was honoured: the proxies were attempted and the honest outcome — they pin no numeric value for either OPEN item — is recorded as PARTIAL, not bypassed with a fabricated bracket. `Q_low` cannot be expressed as a query-volume scalar (the free-tier qualitative limit is capability-driven, not volume-driven); the month-end magnitude has no public proxy. The engine carries both as honest unpinned markers. Per CORR-E10P-8 the §4.3 surface degrades to a centre-only characterization on those two dimensions. This is non-HALT and spec-anticipated. Descriptive posture only.

## Decision-citation block — Q ⊥ FX in the primary spec

> **Reference:** calibration note §6; spec §6.4.
> **Why:** Q is mined from research/build sprints exogenous to FX realized variance; the primary generates Q independently of FX.
> **Relevance to E10 v0.4:** the §4.2 covariance term 2·Cov(Δlog FX, Δlog Q) is ≈ 0 in the primary.
> **Connection to the deliverable:** the behavioral-coupling arm is a pre-committed sensitivity arm only (Phase 5), never a primary rescue.

In [ ]:
# Q is generated independently of FX: two runs against different FX
# paths at the same (currency, month) cell produce identical Q.
engine = NHPPSimulationEngineModule(
    params=calibrated_intensity_parameters())
n_days = 31
fx_calm = tuple(4000.0 + 0.1 * d for d in range(n_days))
fx_wild = tuple(4000.0 + 80.0 * np.sin(d) for d in range(n_days))
traj_calm = engine(calibrated_intensity_parameters(),
                   'COP', 2025, 1, fx_calm)
traj_wild = engine(calibrated_intensity_parameters(),
                   'COP', 2025, 1, fx_wild)
assert traj_calm.daily_query_counts == traj_wild.daily_query_counts
print('VERIFIED: identical Q under a calm and a wild FX path '
      '-> Q is generated independently of FX (spec §6.4).')
print(f'cost differs only through the real FX path: '
      f'calm sum={sum(traj_calm.daily_cost):.2f} USDC, '
      f'wild sum={sum(traj_wild.daily_cost):.2f} USDC')

## Interpretation — calibration summary

The R6 NHPP simulation engine is built and calibrated:

- **Overdispersion:** Cox / doubly-stochastic NHPP, Gamma-mixed sprint-intensity primary, log-normal-mixing sensitivity arm — RATIFIED by the pre-Phase-2 review.
- **Five λ(t) modulations:** four LOCKED and own-trace-anchored; modulation 4's magnitude OPEN.
- **Q-volume range:** centre ~60-130 priceable queries/month, swept as the §4.3 surface domain; `Q_high` = 39,900/mo.
- **Two OPEN items (`Q_low`, month-end magnitude):** the proxy-research gate pinned neither -> PARTIAL per spec §9 (`dispositions/E10.1_proxy_research_PARTIAL.md`); no flat-fill.
- **Q ⊥ FX:** confirmed by construction in the primary.

The simulator is anchored to the genuine NON-FANTASY observed trace — no HALT on anchor grounds. E10 v0.4 makes no inferential beta claim; this notebook is descriptive only. <!-- CHECK_ALLOWLIST: names the firewall-banned posture to forbid it -->